# **LLM Quantization & QLoRA: Practical Guide**
1. **Quantization Kya Hai**? (Simple Bhasha Mein)
Quantization ka matlab hai ek bade model ko "chota" karna bina uski akal (intelligence) khoye.

    **Example:** Jaise ek 4K movie ko 720p mein convert karna taaki wo phone mein kam jagah le, lekin movie puri dikhe.

    **Technical:** Model ke weights (parameters) jo pehle 32-bit (FP32) mein hote hain, unhe hum 4-bit (INT4) mein convert kar dete hain.

2. Hume iski zaroorat kyun hai?
Memory Bachat: Jo model 40GB VRAM mangta hai, wo quantization ke baad sirf 10GB mein chal jata hai.

    **Sasta Setup:** Hum mehnge A100 GPUs ki jagah free Google Colab (T4 GPU) par bhi bade models chala sakte hain.

    **Speed:** Chote numbers ko process karna computer ke liye zyada fast hota hai.

3. Key Terms (Interview/Exams ke liye):
BitsAndBytes: Ye wo main library hai jo 4-bit loading ko handle karti hai.

    **NF4 (Normal Float 4):** Ye quantization ka sabse advanced tarika hai jo kam memory mein bhi model ki quality baniye rakhta hai.

    **Double Quantization:** Isme hum quantization ke extra data ko bhi compress karte hain (Aur bhi zyada memory savings!).

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 42.3 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

print(f"Model Memory Footprint: {model.get_memory_footprint() / 1e6:.2f} MB")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Memory Footprint: 746.77 MB


In [ ]:
prompt = "tell me what is LLM models."

In [ ]:
# Sabse simple method bina kisi template ke
raw_prompt = f"User: {prompt}\nAssistant:"
inputs = tokenizer(raw_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs, # Ye dictionary ko unpack kar deta hai (input_ids + attention_mask)
    max_new_tokens=50
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


User: tell me what is LLM models.
Assistant: LLM models are a type of machine learning models that are specifically designed for legal research. They are used to analyze legal documents, such as court opinions, statutes, and case law, and to identify patterns and relationships between different legal concepts. L


In [ ]:
import torch

memory_used = model.get_memory_footprint() / 1e6
print(f"Quantized Model Memory: {memory_used:.2f} MB")

Quantized Model Memory: 746.77 MB


# Post-Practical Analysis & Key Observations
Memory Efficiency Check
After running the code, notice the Memory Footprint.


**Without Quantization:**

* A 1.1B parameter model in FP32 would take ~4.4 GB.

* With 4-bit Quantization: It takes only ~700-800 MB.


**Conclusion:**

* We reduced the memory usage by nearly 80%!

* Inference Quality
Even though we compressed the model's weights to 4-bit:

* The model still provides logical and coherent answers.

This proves that LLMs have high redundancy, meaning we can remove precision without losing much intelligence.

#Second Practical: Scaling to a 7-Billion Parameter Model

**Objective**

* To demonstrate that Quantization is a "Superpower." We will load a 7-Billion parameter model (Mistral-7B), which normally requires 28GB+ VRAM, on our free 16GB Google Colab GPU.


**The Model: Mistral-7B-v0.3 (4-bit)
Total Parameters: 7 Billion**

* Memory without Quantization: ~28 GB (Crash on Colab)

* Memory with 4-bit Quantization: ~5.5 GB (Runs Smoothly)

In [ ]:
'unsloth/mistral-7b-v0.3-bnb-4bit'